# 🚦 Crosswalk Violation System — Google Colab Runner

## Recommended workflow (VS Code ↔ Colab)

1. Install **Google Drive for Desktop** on Windows and sign in.
2. Move (or keep) your project folder inside **My Drive** so it syncs automatically.
3. Edit all code in **VS Code** — changes sync to Drive in seconds.
4. Open this notebook in **Colab**, set runtime to GPU, and run the cells below.

**Runtime:** `Runtime → Change runtime type → T4 GPU → Save`

| Cell | What it does |
|------|--------------|
| 1 | Verify GPU is active |
| 2 | Mount Google Drive — **update `DRIVE_PROJECT_PATH`** |
| 3 | Install dependencies |
| 4 | Set video path — **update `VIDEO_FILENAME`** |
| 5 | Write `.env` config for GPU |
| 6 | Run detection — saves `output_annotated.mp4` back to Drive |
| 7 | Preview output frames inline |
| 8 | Confirm output location in Drive |

In [ ]:
# ── Cell 1: Verify GPU runtime ─────────────────────────────────────────────
import subprocess, shutil

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
else:
    print('⚠️ nvidia-smi not found — GPU runtime is NOT active!')
    print('Go to: Runtime → Change runtime type → T4 GPU → Save → then reconnect.')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ CUDA not available. Stop here and switch to a GPU runtime before continuing.')

In [ ]:
# ── Cell 2: Mount Google Drive and set project root ─────────────────────────
# Prerequisites:
#   - Install "Google Drive for Desktop" on Windows
#   - Your project folder must be inside My Drive (it syncs automatically)
#   - Edit code in VS Code locally — changes appear in Colab after sync
#
# Update DRIVE_PROJECT_PATH if your folder is in a subfolder of Drive.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/00016886_Crosswalk_Violation'  # <-- update if needed

import os, sys
assert os.path.isdir(DRIVE_PROJECT_PATH), (
    f'Project folder not found: {DRIVE_PROJECT_PATH}\n'
    f'Check that Google Drive for Desktop is syncing and the path is correct.'
)
os.chdir(DRIVE_PROJECT_PATH)
sys.path.insert(0, DRIVE_PROJECT_PATH)
PROJECT_ROOT = DRIVE_PROJECT_PATH
print(f'Project root: {PROJECT_ROOT}')
print('Contents:', os.listdir('.'))

In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
# opencv-python-headless is used instead of opencv-python (no display server in Colab).
# The headless build can still open video files and do all image processing.
import subprocess, sys

pkgs = [
    "ultralytics", "lap", "easyocr",
    "flask", "werkzeug", "gunicorn", "python-dotenv",
    "sqlalchemy", "alembic", "pillow", "reportlab",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

# Replace opencv-python with the headless variant (avoids display-server errors)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "opencv-python"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)

import ultralytics, cv2
print(f"Ultralytics : {ultralytics.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print("All dependencies installed.")

In [ ]:
# ── Cell 4: Set video path from Google Drive ────────────────────────────────
# Place your video inside the Videos/ subfolder of the project in Drive,
# then set VIDEO_FILENAME below.

import os

VIDEO_FILENAME = 'v2.mp4'  # <-- change this to your video file name

VIDEO_PATH = os.path.join(PROJECT_ROOT, 'Videos', VIDEO_FILENAME)
assert os.path.isfile(VIDEO_PATH), (
    f'Video not found: {VIDEO_PATH}\n'
    f'Make sure the file is in the Videos/ folder and Drive has finished syncing.'
)
print(f'Video ready at: {VIDEO_PATH}')

In [ ]:
# ── Cell 5: Configure environment for GPU ───────────────────────────────────
import os

env_content = f"""VIDEO_PATH={VIDEO_PATH}
DETECTION_MODEL_PATH=yolov8x.pt
IMAGE_SIZE=1280
DETECTION_CONFIDENCE=0.25
YOLO_DEVICE=0
YOLO_HALF=1
PIPELINE_WORKERS=2
DATABASE_URL=sqlite:///crosswalk_violations.db
SQLITE_FALLBACK_URL=sqlite:///crosswalk_violations.db
LLM_PROVIDER=mock
ENABLE_STABILIZATION=0
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('.env written:')
print(env_content)

In [ ]:
# ── Cell 6: Run headless detection — saves annotated output video ───────────
import sys, os
sys.path.insert(0, 'src')

from dotenv import load_dotenv
load_dotenv('.env')

import cv2
import numpy as np
from pathlib import Path

from config import (
    CONF_THRESHOLD, DETECTION_CLASSES, IMG_SIZE, MODEL_PATH, VIDEO_PATH, settings
)
from detector.yolo_detector import YOLODetector
from detector.tracker import IDMerger, PedestrianTrack, VehicleTrack, apply_cross_class_nms
from geometry.crosswalk import CrosswalkZone
from logic.violation import (
    ENTRY_EVAL_WINDOW_FRAMES, TRACK_RESET_FRAMES,
    check_violation, compute_approach_axis, get_polygon_midline, update_pedestrian_state
)
from schemas import ViolationEvent
from services.pipeline import EnforcementPipeline
from vision.draw import draw_box

_AMBER_BGR = (11, 158, 245)

def draw_zone_overlay(frame, polygon, color, alpha=0.2):
    overlay = frame.copy()
    cv2.fillPoly(overlay, [polygon], color)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    cv2.polylines(frame, [polygon], True, color, 2)

# ── Load polygon ─────────────────────────────────────────────────────────────
import json
polygon_path = Path(settings.runtime.polygon_path)
assert polygon_path.exists(), (
    f'Polygon file not found: {polygon_path}. '
    f'Make sure crosswalk_polygon.json is in the project folder.'
)
polygon_data = json.loads(polygon_path.read_text())
polygon = polygon_data if isinstance(polygon_data, list) else polygon_data.get('polygon', [])
assert len(polygon) >= 3, 'Polygon must have at least 3 points'
np_polygon = np.array(polygon, dtype=np.float32)
print(f'Polygon loaded: {len(polygon)} points')

# ── Setup ─────────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'Cannot open video: {VIDEO_PATH}'
fps    = cap.get(cv2.CAP_PROP_FPS) or 30
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {width}x{height} @ {fps:.1f} fps, {total} frames')

OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'output_annotated.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

crosswalk    = CrosswalkZone(polygon)
upper_poly, lower_poly = crosswalk.get_split_polygons(ratio=settings.runtime.split_ratio)

pw = float(np_polygon[:, 0].max() - np_polygon[:, 0].min())
ph = float(np_polygon[:, 1].max() - np_polygon[:, 1].min())
default_approach_axis   = 'from_bottom' if ph >= pw else 'from_left'
default_polygon_midline = get_polygon_midline(np_polygon, default_approach_axis)

print(f'Loading model: {MODEL_PATH} on GPU...')
detector  = YOLODetector(MODEL_PATH, DETECTION_CLASSES, CONF_THRESHOLD, IMG_SIZE)
id_merger = IDMerger(proximity_px=40.0, min_frames=3)
enforcement_pipeline = EnforcementPipeline(settings)

ped_tracks: dict           = {}
veh_tracks: dict           = {}
vehicles_in_polygon: set   = set()
active_violation_cars: set = set()
triggered_pairs: set       = set()
violation_count            = 0
frame_index                = 0

print(f'Processing {total} frames...')
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_index += 1

        if frame_index % 100 == 0:
            print(f'  Frame {frame_index}/{total} | violations: {violation_count}')

        results = detector.detect(frame)

        crosswalk.draw(frame)
        crosswalk.draw_half_split(frame, ratio=settings.runtime.split_ratio)
        draw_zone_overlay(frame, upper_poly, (255, 0, 0), alpha=0.15)
        draw_zone_overlay(frame, lower_poly, (0, 255, 0), alpha=0.15)

        cv2.putText(frame, f'P:{len(ped_tracks)} V:{len(veh_tracks)} VIOLATIONS:{violation_count}',
                    (16, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        if results and results[0].boxes.id is not None:
            boxes   = results[0].boxes.xyxy.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy().astype(int)
            ids     = results[0].boxes.id.cpu().numpy().astype(int)
            confs   = results[0].boxes.conf.cpu().numpy()

            boxes, classes, ids, confs = apply_cross_class_nms(boxes, classes, ids, confs, 0.5)
            ids = id_merger.update(ids, boxes)

            current_ped_ids: set = set()
            current_veh_ids: set = set()
            newly_in_polygon: set = set()

            for box, cls, obj_id in zip(boxes, classes, ids):
                obj_id = int(obj_id)
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0

                if cls == 0:
                    current_ped_ids.add(obj_id)
                    if obj_id not in ped_tracks:
                        ped_tracks[obj_id] = PedestrianTrack(track_id=obj_id)
                    pt = ped_tracks[obj_id]
                    pt.prev_centroid = pt.centroid
                    pt.centroid = (cx, cy)
                    pt.bbox = (x1, y1, x2, y2)
                    if pt.prev_centroid is not None:
                        pt.velocity_history.append((cx - pt.prev_centroid[0], cy - pt.prev_centroid[1]))
                    pt.frames_outside_count = 0
                    update_pedestrian_state(pt, np_polygon, default_polygon_midline, default_approach_axis, frame_index)
                else:
                    current_veh_ids.add(obj_id)
                    if obj_id not in veh_tracks:
                        veh_tracks[obj_id] = VehicleTrack(track_id=obj_id)
                    vt = veh_tracks[obj_id]
                    vt.prev_centroid = vt.centroid
                    vt.centroid = (cx, cy)
                    if vt.prev_centroid is not None:
                        vt.velocity_history.append((cx - vt.prev_centroid[0], cy - vt.prev_centroid[1]))
                    vt.centroid_history.append((cx, cy))
                    inside = crosswalk.intersects_box(box, min_ratio=0.02)
                    if inside:
                        newly_in_polygon.add(obj_id)
                        if obj_id not in vehicles_in_polygon:
                            approach_axis = compute_approach_axis(vt.centroid_history, np_polygon)
                            vt.approach_axis = approach_axis
                            vt.polygon_midline = get_polygon_midline(np_polygon, approach_axis)
                            vt.polygon_entry_frame = frame_index
                            vt.pre_entry_velocity_snapshot = list(vt.velocity_history)
                            triggered_pairs -= {p for p in triggered_pairs if p[0] == obj_id}
                            active_violation_cars.discard(obj_id)
                    else:
                        vt.polygon_entry_frame = None
                        active_violation_cars.discard(obj_id)

            vehicles_in_polygon = newly_in_polygon

            for gone_id in list(ped_tracks):
                if gone_id not in current_ped_ids:
                    ped_tracks[gone_id].frames_outside_count += 1
                    if ped_tracks[gone_id].frames_outside_count > TRACK_RESET_FRAMES:
                        del ped_tracks[gone_id]
            for gone_id in list(veh_tracks):
                if gone_id not in current_veh_ids:
                    triggered_pairs -= {p for p in triggered_pairs if p[0] == gone_id}
                    active_violation_cars.discard(gone_id)
                    del veh_tracks[gone_id]

            for box, cls, obj_id in zip(boxes, classes, ids):
                obj_id = int(obj_id)
                x1, y1, x2, y2 = box
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                if cls != 0:
                    vt = veh_tracks.get(obj_id)
                    if (vt is not None and vt.polygon_entry_frame is not None
                            and (frame_index - vt.polygon_entry_frame) <= ENTRY_EVAL_WINDOW_FRAMES):
                        for ped_id, pt in ped_tracks.items():
                            pair = (obj_id, ped_id)
                            if pair in triggered_pairs:
                                continue
                            v = check_violation(
                                car_track=vt, ped_track=pt, polygon=np_polygon,
                                frame_number=frame_index,
                                approach_axis=vt.approach_axis or default_approach_axis,
                                polygon_midline=vt.polygon_midline or default_polygon_midline,
                            )
                            if v is not None:
                                triggered_pairs.add(pair)
                                active_violation_cars.add(obj_id)
                                violation_count += 1
                                print(f'  *** VIOLATION at frame {frame_index}: car {obj_id} / ped {ped_id} — {v.violation_type}')
                                from main import build_event
                                event = build_event(
                                    frame_index=frame_index, box=box, car_id=obj_id,
                                    violation=v, polygon=polygon, ped_track=pt, veh_track=vt
                                )
                                enforcement_pipeline.submit_violation(frame.copy(), event)

                box_color = ((0, 0, 255) if (cls != 0 and obj_id in active_violation_cars)
                             else (0, 255, 255) if (cls != 0 and obj_id in vehicles_in_polygon)
                             else (0, 255, 0))
                draw_box(frame, box, 'person' if cls == 0 else 'vehicle', box_color)

                if cls != 0 and obj_id in active_violation_cars:
                    cv2.putText(frame, 'VIOLATION', (cx, cy - 25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

        writer.write(frame)

finally:
    enforcement_pipeline.shutdown()
    cap.release()
    writer.release()
    print(f'Done. {frame_index} frames processed, {violation_count} violations detected.')
    print(f'Output saved to: {OUTPUT_PATH}')

In [ ]:
# ── Cell 7: Preview output frames inline ────────────────────────────────────
from IPython.display import Image, display
import cv2

cap_out = cv2.VideoCapture(OUTPUT_PATH)
total_out = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))

# Show 6 evenly-spaced frames from the output
sample_frames = [int(total_out * i / 5) for i in range(6)]

for idx in sample_frames:
    cap_out.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap_out.read()
    if not ret:
        continue
    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
    display(Image(data=buf.tobytes()))
    print(f'Frame {idx}/{total_out}')

cap_out.release()

In [ ]:
# ── Cell 8: Confirm output saved to Google Drive ────────────────────────────
# The output video is already written directly to your Drive project folder.
# No download needed — open it from Windows via Google Drive for Desktop.

import os

if os.path.isfile(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print('✅ Output saved to Google Drive:')
    print(f'   {OUTPUT_PATH}')
    print(f'   Size: {size_mb:.1f} MB')
    print()
    print('Open it on Windows at:')
    print('   Google Drive (G:) → My Drive → 00016886_Crosswalk_Violation → output_annotated.mp4')
else:
    print(f'⚠️ Output not found at: {OUTPUT_PATH}')
    print('Check that Cell 6 completed without errors.')

---

## Section B — Live Admin Panel Demo (GPU-accelerated)

> **Use this section to demo the full Flask web app to your professors.**
> It runs the admin panel with real-time YOLO detection on Colab's T4 GPU
> and exposes it publicly via a **Cloudflare Quick Tunnel** — no account needed.

### Why this is fast (and why RunPod was slow)

| Approach | Frame path | Latency |
|---|---|---|
| **RunPod** (what you had) | PC camera → internet → RunPod GPU → internet → browser | 200–500 ms/frame |
| **This setup** | Drive video read locally on Colab → GPU → internet → browser | 20–50 ms/frame |

The demo videos (`v2.mp4`, etc.) live in your Google Drive, which is **mounted directly on Colab** — Colab reads them as if they were a local file. No frames travel over the network inbound. Only the final annotated MJPEG stream travels out to your browser.

### Prerequisites
- Complete **Cells 1, 2, 3** above (GPU verified + Drive mounted + deps installed)
- Your `Videos/` folder must be synced to Google Drive on your PC

### No account required
Cell B2 downloads `cloudflared` — a single binary from Cloudflare. Running it creates
a public `trycloudflare.com` HTTPS URL instantly with no login or token.

### Optional: connect your actual local webcam
If you want live camera instead of the demo video, run `camera_bridge.py` on your
Windows PC (see the file in the project root) and paste its cloudflared URL into
`CAMERA_SOURCE` in Cell B3.

In [ ]:
# ── Cell B2: Download cloudflared tunnel binary ───────────────────────────────
# cloudflared (by Cloudflare) creates a public HTTPS URL with ZERO account setup.
# Each session gets a fresh *.trycloudflare.com URL — that's fine for a demo.

import subprocess, os

if not os.path.isfile('cloudflared'):
    print("Downloading cloudflared...")
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', 'cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)
    print("✅ cloudflared ready")
else:
    print("✅ cloudflared already downloaded")

# Quick smoke-test
result = subprocess.run(['./cloudflared', 'version'], capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
# ── Cell B3: Write .env for live Flask mode ─────────────────────────────────
#
# CAMERA_SOURCE options:
#   "Videos/v2.mp4"                                     ← demo video (default, no setup needed)
#   "https://YOUR-PC-NGROK.ngrok-free.app/video"        ← live local webcam via camera_bridge.py
#
import os

CAMERA_SOURCE = "Videos/v2.mp4"   # change to ngrok URL if using camera_bridge.py

env_content = f"""SECRET_KEY=colab-demo-secret-key-change-for-prod
FLASK_ENV=production
CAMERA_SOURCE={CAMERA_SOURCE}
DEMO_VIDEO_SOURCE=Videos/v2.mp4
VIDEO_PATH=Videos/v2.mp4
DETECTION_MODEL_PATH=yolov8n.pt
PLATE_MODEL_PATH=models/plate_detector.pt
PLATE_CONFIDENCE=0.25
PLATE_CLASSES=0
IMAGE_SIZE=960
DETECTION_CONFIDENCE=0.25
PIPELINE_WORKERS=2
DATABASE_URL=sqlite:///crosswalk_violations.db
SQLITE_FALLBACK_URL=sqlite:///crosswalk_violations.db
LLM_PROVIDER=mock
AUTHORITY_NAME=WIUT Traffic Enforcement Unit
LOCATION_NAME=Crosswalk A
LOCATION_CODE=CW-A-01
DEFAULT_FINE_AMOUNT=150000
LOCATION_LATITUDE=41.2963
LOCATION_LONGITUDE=69.2798
"""

with open('.env', 'w') as f:
    f.write(env_content.strip())

print(f"✅ .env written for live Flask mode")
print(f"   Camera source: {CAMERA_SOURCE}")

In [ ]:
# ── Cell B4: Start Flask + create Cloudflare tunnel (no account needed) ──────
#
# HOW LATENCY IS SOLVED HERE:
#   The demo video (v2.mp4) lives inside your Google Drive, which is mounted at
#   /content/drive. Colab reads it locally — zero network round-trip for frames.
#   Only the final annotated MJPEG stream travels over the tunnel to your browser.
#   GPU does inference in <20 ms per frame.
#
# Cloudflare Quick Tunnel requires NO account — a public URL is created instantly.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, threading, time, sys, os, re

flask_log  = []
flask_proc = None
tunnel_proc = None
public_url  = None

# ── Start Flask ───────────────────────────────────────────────────────────────
def _run_flask():
    global flask_proc
    env = os.environ.copy()
    env['PYTHONPATH'] = os.getcwd() + '/src:' + env.get('PYTHONPATH', '')
    env['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
    flask_proc = subprocess.Popen(
        [sys.executable, 'app.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
        bufsize=1,
    )
    for line in iter(flask_proc.stdout.readline, ''):
        flask_log.append(line.rstrip())
        if any(kw in line for kw in ('Running on', 'Serving Flask', 'Traceback', ' Error')):
            print(line, end='')

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()

print("⏳ Starting Flask + loading YOLO model on GPU...")
print("   First run downloads yolov8n.pt (~6 MB) — allow up to 40 seconds.")

# Poll until Flask says it's ready (or timeout after 60 s)
for i in range(60):
    time.sleep(1)
    ready = any('Running on' in l or 'Serving Flask' in l for l in flask_log)
    error = any('Traceback' in l or 'ImportError' in l for l in flask_log)
    if ready:
        print(f"   Flask ready after {i+1} s")
        break
    if error:
        print("   ⚠️  Flask hit an error — run Cell B5 to see the full log")
        break
    if i % 10 == 9:
        print(f"   Still loading... ({i+1} s)")
else:
    print("   Timeout — Flask may still be starting. Run Cell B5 to check.")

# ── Start Cloudflare Quick Tunnel ─────────────────────────────────────────────
def _start_tunnel():
    global tunnel_proc, public_url
    tunnel_proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in tunnel_proc.stdout:
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group()
            break  # URL found — process stays alive in background

tunnel_thread = threading.Thread(target=_start_tunnel, daemon=True)
tunnel_thread.start()

print("⏳ Creating Cloudflare tunnel (no account needed)...")
for i in range(30):
    time.sleep(1)
    if public_url:
        break
    if i % 10 == 9:
        print(f"   Waiting for tunnel URL... ({i+1}s)")
else:
    public_url = "(tunnel URL not captured — run Cell B5 to check logs)"

border = "═" * 66
print(f"""
╔{border}╗
║{'CROSSWALK VIOLATION SYSTEM  —  LIVE GPU DEMO':^66}║
╠{border}╣
║  {'Admin panel URL:':<20}{(public_url + '/admin/login'):<46}║
║  {'Username:':<20}{'admin':<46}║
║  {'Password:':<20}{'admin1234':<46}║
╠{border}╣
║  After login → Live Camera → CONNECT → watch GPU detection live  ║
╚{border}╝
""")

In [ ]:
# ── Cell B5: Check Flask startup logs ───────────────────────────────────────
# Run this if the admin panel is not loading — shows what Flask printed on startup.
print("=== Flask startup log (last 30 lines) ===")
print("\n".join(flask_log[-30:]) if flask_log else "No log lines yet — wait a few seconds and retry.")

In [ ]:
# ── Cell B6: Stop the server when demo is finished ──────────────────────────
# Run this cell when done to release GPU memory and close the tunnel.
if 'tunnel_proc' in dir() and tunnel_proc and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    print("Tunnel stopped.")
if 'flask_proc' in dir() and flask_proc and flask_proc.poll() is None:
    flask_proc.terminate()
    print("Flask stopped.")
print("✅ Server stopped. GPU memory released.")